# 02 - Joins and Aggregations

Objetivo: juntar orders, customers e products para criar metricas de receita.

In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

java_home = Path("/usr/local/opt/openjdk@17")
if not java_home.exists():
    java_home = Path("/opt/homebrew/opt/openjdk@17")

os.environ["JAVA_HOME"] = str(java_home)
os.environ["PATH"] = f"{java_home / 'bin'}:{os.environ['PATH']}"
os.environ.setdefault("SPARK_LOCAL_IP", "127.0.0.1")

print(f"Python: {sys.executable}")
print(f"JAVA_HOME: {os.environ['JAVA_HOME']}")

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, round as spark_round, sum as spark_sum

In [ ]:
spark = (
    SparkSession.builder.appName("notebook-02-joins-and-aggregations")
    .master("local[*]")
    .getOrCreate()
)


In [ ]:
def read_csv(name):
    return (
        spark.read.option("header", True)
        .option("inferSchema", True)
        .csv(str(PROJECT_ROOT / f"data/raw/{name}.csv"))
    )


orders = read_csv("orders")
customers = read_csv("customers")
products = read_csv("products")

## Join entre tabelas

Esta tabela enriquecida e parecida com uma Silver/Gold table simples.

In [ ]:
sales = (
    orders.filter(col("status") == "delivered")
    .join(customers, on="customer_id", how="left")
    .join(products, on="product_id", how="left")
    .withColumn("revenue", spark_round(col("quantity") * col("unit_price"), 2))
)

sales.select(
    "order_id",
    "name",
    "country",
    "product_name",
    "category",
    "quantity",
    "revenue",
).show(truncate=False)

## Receita por pais

In [ ]:
sales.groupBy("country").agg(
    spark_round(spark_sum("revenue"), 2).alias("total_revenue")
).orderBy(col("total_revenue").desc()).show(truncate=False)

## Receita por categoria

In [ ]:
sales.groupBy("category").agg(
    spark_round(spark_sum("revenue"), 2).alias("total_revenue")
).orderBy(col("total_revenue").desc()).show(truncate=False)

## Para praticar

Cria uma tabela com receita por cliente. Colunas sugeridas: `customer_id`, `name`, `country`, `total_revenue`.